# Acervo que Fala — Notebook 05: bake-off de redator (Qwen × Gemma)

**Projeto final** · Inteligência Artificial Generativa & Large Language Models (ICA/PUC-Rio) · Eduardo Tosto

**Bake-off** é um teste comparativo lado a lado: as mesmas entradas em dois candidatos, julgadas pelos mesmos critérios. Aqui, o candidato desafiante é o **Gemma 3 12B** (Google, aberto, com fama de prosa mais natural em português) escrevendo a **redação** — a etapa de texto do pipeline. Tudo o mais fica idêntico ao Notebook 04 v5:

- as **mesmas observações visuais** (reaproveitadas do resultado salvo — nenhuma imagem é reprocessada, por isso esta sessão é curta);
- o **mesmo registro** do museu, a **mesma rubrica v1.1** e o **mesmo prompt v8** — lido de dentro do resultado do Notebook 04, garantindo que nem uma vírgula seja diferente;
- a **mesma verificação automática**.

Se o Gemma escrever melhor sob as mesmas regras, ele assume a redação. Se não, o Qwen fica — e a escolha terá sido feita com dados, não com fama de benchmark.

*Metodologia: projeto construído por um designer com LLMs como suporte (vibe coding) — cada célula explicada.*

### Como rodar
**Pré-requisito: o Notebook 04 v5 precisa ter rodado antes** (este notebook lê o resultado dele).
1. **Ambiente de execução → Alterar o tipo → GPU T4** · 2. **Executar tudo** · 3. Tempo: **~20–30 min** (download do modelo + 20 gerações de texto).

In [ ]:
# Etapa 1 — Instalação (Pillow travada, regra da casa) + checagem do ambiente
import PIL
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers pillow=={PIL.__version__}
import torch, transformers
print(f"transformers {transformers.__version__} | GPU: {torch.cuda.is_available()}")
print("ambiente íntegro ✓")

## Etapa 2 — Carregar os insumos do Notebook 04 v5

O resultado do Notebook 04 v5 carrega tudo o que a redação precisa: as observações visuais, os registros completos e o **próprio prompt v8** (o prompt viaja junto com o resultado — assim o bake-off usa exatamente o mesmo texto de instruções, sem risco de cópia divergente). A rubrica v1.1 e os embeddings do RAG são os mesmos do pipeline.

In [ ]:
import json
from google.colab import drive
from sentence_transformers import SentenceTransformer, util

drive.mount("/content/drive")
PROJETO = "/content/drive/MyDrive/00_IA/GenAI & LLMs - PUC/Projeto_LLM"

with open(f"{PROJETO}/resultados/04_pipeline_completo_v5.json", encoding="utf-8") as f:
    base = json.load(f)
objetos = [dict(i) for i in base["itens"]]
PROMPT_REDACAO_V8 = base["prompt_redacao_v8"]  # o MESMO prompt, lido do resultado
print(f"insumos: {len(objetos)} objetos do {base['notebook']} (redator original: {base['modelo']})")

with open(f"{PROJETO}/dados/rubrica_v1_1.json", encoding="utf-8") as f:
    rubrica = json.load(f)
trechos = rubrica["trechos"]

embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
vetores = embedder.encode([t["texto"] for t in trechos], convert_to_tensor=True)

def recuperar(consulta, k=3):
    v = embedder.encode(consulta, convert_to_tensor=True)
    scores = util.cos_sim(v, vetores)[0]
    achados = []
    for i in scores.argsort(descending=True).tolist():
        if trechos[i]["categoria"] == "geral":
            continue
        achados.append(trechos[i])
        if len(achados) == k:
            break
    return achados

print(f"rubrica {rubrica['versao']}: {len(trechos)} trechos indexados ✓")

## Etapa 3 — Carregar o desafiante: Gemma 3 12B

Usamos a versão da comunidade Unsloth **já quantizada em 4-bit** (`unsloth/gemma-3-12b-it-unsloth-bnb-4bit`): o download cai de ~24GB para ~8GB, cabe na T4, e dispensa o cadastro/licença que o repositório oficial do Google exige. O Gemma 3 12B também é multimodal, mas aqui ele trabalha **só com texto** — recebe a observação pronta, como o pipeline manda.

In [ ]:
import re
from transformers import AutoProcessor, AutoModelForImageTextToText

REDATOR = "unsloth/gemma-3-12b-it-unsloth-bnb-4bit"
modelo = AutoModelForImageTextToText.from_pretrained(REDATOR, device_map="auto", torch_dtype=torch.float16)
processador = AutoProcessor.from_pretrained(REDATOR)

def gerar_texto(texto, max_tokens=700):
    conversa = [{"role": "user", "content": [{"type": "text", "text": texto}]}]
    entradas = processador.apply_chat_template(
        conversa, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(modelo.device)
    with torch.no_grad():
        saida = modelo.generate(**entradas, max_new_tokens=max_tokens)
    return processador.decode(saida[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def extrair_json(texto):
    texto = re.sub(r"^```(json)?|```$", "", texto.strip(), flags=re.MULTILINE).strip()
    inicio, fim = texto.find("{"), texto.rfind("}")
    return json.loads(texto[inicio:fim + 1])

print("Gemma carregado ✓")

## Etapa 4 — Redação com o Gemma (~15 min)

O mesmo laço do Notebook 04: para cada objeto, recupera as diretrizes da rubrica, monta o prompt v8 com a observação e o registro, e pede o JSON estruturado. A única variável que muda no experimento inteiro é o modelo que escreve.

In [ ]:
for n, obj in enumerate(objetos, 1):
    registro_txt = "\n".join(f"{k}: {v}" for k, v in obj["registro"].items() if v)
    consulta = f"{obj['titulo']} ({obj['registro']['Categoria']}). {obj['observacao'][:250]}"
    achados = recuperar(consulta)
    obj["diretrizes_usadas"] = [t["id"] for t in achados]
    prompt = PROMPT_REDACAO_V8.format(
        observacao=obj["observacao"],
        registro=registro_txt,
        diretrizes="\n".join(f"- {t['texto']}" for t in achados),
    )
    resposta = gerar_texto(prompt)
    try:
        saida = extrair_json(resposta)
        obj["alt_text"] = saida["alt_text"]
        obj["descricao_objeto"] = saida["descricao_objeto"]
        obj["flags"] = saida.get("flags", [])
        obj["json_valido"] = True
    except Exception:
        obj["alt_text"], obj["descricao_objeto"], obj["flags"] = resposta, "", []
        obj["json_valido"] = False
    print(f"[{n}/{len(objetos)}] {obj['titulo']}: {obj['alt_text'][:80]}... | flags: {len(obj['flags'])}")

## Etapa 5 — A mesma verificação automática do Notebook 04

Mesmas checagens, mesmo juiz — para o placar ser comparável linha a linha com o resultado do Qwen.

In [ ]:
TERMOS_ARTEFATO = ["cartela", "paleta", "numeração", "marcação", "etiqueta", "régua", "suporte"]
ABERTURAS_ETIQUETA = ["o objeto é", "trata-se de"]
TERMOS_AUSENCIA = ["não há", "sem etiqueta", "sem sinais", "sem evidência", "sem artefatos", "sem marcas"]
TERMOS_FOTO = ["posicionad", "inclinad", "enquadr", "fotografia", "na imagem", "da imagem"]
TERMOS_ESPECULACAO = ["sugere", "sugerindo", "parece ", "parecendo", "possivelmente"]
FRASES_VAZIAS = ["porte médio", "uso prático", "uso frequente", "sinais de uso", "forma funcional", "forma é funcional"]

def tem_atribuicao(texto):
    t = texto.lower()
    return "registro" in t or "catálogo" in t or "catalogo" in t

for obj in objetos:
    p = []
    if not obj["json_valido"]:
        p.append("JSON inválido")
    povo = obj["registro"]["Povo"]
    a = obj["alt_text"].lower()
    if povo and povo.split()[0].lower() not in a:
        p.append(f"povo '{povo}' ausente do alt")
    for termo in TERMOS_ARTEFATO:
        if termo in a:
            p.append(f"artefato no alt ('{termo}')")
    if "fundo" in a:
        p.append("'fundo' no alt (fundo de estúdio? base da peça vira material)")
    if len(obj["alt_text"].split()) > 30:
        p.append(f"{len(obj['alt_text'].split())} palavras")
    d = obj["descricao_objeto"].lower().strip()
    if obj["descricao_objeto"]:
        if not tem_atribuicao(obj["descricao_objeto"]):
            p.append("nível 2 sem atribuição ao registro")
        if any(d.startswith(ab) for ab in ABERTURAS_ETIQUETA):
            p.append("nível 2 abre com frase-etiqueta")
        if "a função é" in d:
            p.append("frase-etiqueta ('a função é')")
        if "aquisição em" in d:
            p.append("'foi aquisição em' (usar 'adquirido em')")
        for termo in TERMOS_ARTEFATO:
            if termo in d:
                p.append(f"artefato no nível 2 ('{termo}')")
        for termo in TERMOS_AUSENCIA:
            if termo in d:
                p.append(f"afirmação de ausência ('{termo}')")
        for termo in TERMOS_FOTO:
            if termo in d:
                p.append(f"foto no nível 2 ('{termo}')")
    for nome, texto in [("alt", a), ("nível 2", d)]:
        for termo in TERMOS_ESPECULACAO:
            if termo in texto:
                p.append(f"especulação no {nome} ('{termo.strip()}')")
        for termo in FRASES_VAZIAS:
            if termo in texto:
                p.append(f"frase vazia no {nome} ('{termo}')")
    for f in obj["flags"]:
        det = f["detalhe"].lower()
        if f["tipo"] == "artefato_estudio" and "fundo" in det and not any(t in det for t in TERMOS_ARTEFATO):
            p.append("flag de fundo (ruído — fundo de estúdio não é artefato)")
        if any(t in det for t in ["sem ", "não há"]):
            p.append("flag afirmando ausência")
    obj["problemas"] = p
    status = "✓" if not p else "⚠ " + "; ".join(p)
    print(f"{obj['id']} {obj['titulo'][:30]:30} {status}")

print(f"\nTotal Gemma: {sum(1 for o in objetos if not o['problemas'])}/{len(objetos)} objetos sem problemas | {sum(len(o['flags']) for o in objetos)} flags")

In [ ]:
# Etapa 6 — Salvar no Drive
resultado = {
    "notebook": "05_bakeoff_redator_v1",
    "redator": REDATOR,
    "observacoes_de": base["notebook"],
    "embedding": "Qwen/Qwen3-Embedding-0.6B",
    "rubrica_versao": rubrica["versao"],
    "prompt_redacao_v8": PROMPT_REDACAO_V8,
    "itens": [
        {"id": o["id"], "titulo": o["titulo"], "registro": o["registro"],
         "observacao": o["observacao"], "alt_text": o["alt_text"],
         "descricao_objeto": o["descricao_objeto"], "flags": o["flags"],
         "diretrizes_usadas": o["diretrizes_usadas"],
         "json_valido": o["json_valido"], "problemas": o["problemas"]}
        for o in objetos
    ],
}
destino = f"{PROJETO}/resultados/05_bakeoff_gemma.json"
with open(destino, "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=2)
print(f"salvo no Drive ✓  {destino}")

---

## Fim — o que fazer agora

Avise o Claude que o bake-off terminou — ele busca os dois resultados no Drive e monta o **placar comparativo**: checagens automáticas lado a lado + página de revisão com os textos dos dois modelos par a par, para o julgamento editorial final ser seu.

**O que este notebook prova:** a escolha do modelo redator feita com o instrumento de avaliação do próprio projeto — mesmas entradas, mesmas regras, mesmo juiz. **Critério de decisão:** o Gemma só assume se vencer nas checagens automáticas E no seu julgamento editorial; empate mantém o Qwen (um modelo só no pipeline é mais simples).